In [1]:
import sympy as smp
import numpy as np

In [2]:
t, m, R, g, l, lm = smp.symbols(r"t m R g l \lambda", real=True)
r, the, phi = smp.symbols(r"r \theta \phi", cls=smp.Function)
r = r(t)
the = the(t)

def dt(x):
    return smp.diff(x), smp.diff(smp.diff(x))

r_d, r_dd = dt(r)
the_d, the_dd = dt(the)

In [3]:
v = smp.Matrix([r_d, r*the_d])

In [4]:
T = 1/2 * m * v.dot(v)
U = m * g * r * smp.cos(the)
L = T - U
L

-g*m*r(t)*cos(\theta(t)) + 0.5*m*(r(t)**2*Derivative(\theta(t), t)**2 + Derivative(r(t), t)**2)

In [5]:
def lagrange_eq(L, variables, R=0):
    lagrange_equations = []

    for q in variables:
        q_dot = smp.diff(q, t)
        LE = smp.diff(smp.diff(L, q_dot), t) - smp.diff(L, q) + smp.diff(R, q_dot)
        LE = LE.simplify()
        lagrange_equations.append(LE)
        
    return tuple(lagrange_equations)

In [6]:
LEr, LEthe = lagrange_eq(L, (r, the))

In [7]:
LEr

1.0*m*(g*cos(\theta(t)) - r(t)*Derivative(\theta(t), t)**2 + Derivative(r(t), (t, 2)))

In [8]:
LEthe

m*(-g*sin(\theta(t)) + 1.0*r(t)*Derivative(\theta(t), (t, 2)) + 2.0*Derivative(\theta(t), t)*Derivative(r(t), t))*r(t)

# Constraints

The constraint the ball stays anchored to the hoop. So the position $r$ is always equal to the radius $R$.

$$ r = R $$
$$ r - R = 0 $$

In [9]:
system = smp.Matrix(
    [
        LEr + lm,
        LEthe,
    ]
)
system

Matrix([
[                      \lambda + 1.0*m*(g*cos(\theta(t)) - r(t)*Derivative(\theta(t), t)**2 + Derivative(r(t), (t, 2)))],
[m*(-g*sin(\theta(t)) + 1.0*r(t)*Derivative(\theta(t), (t, 2)) + 2.0*Derivative(\theta(t), t)*Derivative(r(t), t))*r(t)]])

Substitue the constraints

In [10]:
system = system.subs({r: R, r_d: 0, r_dd: 0})
system

Matrix([
[\lambda + 1.0*m*(-R*Derivative(\theta(t), t)**2 + g*cos(\theta(t)))],
[       R*m*(1.0*R*Derivative(\theta(t), (t, 2)) - g*sin(\theta(t)))]])

In [11]:
smp.solve(system[1], the_dd)[0]

g*sin(\theta(t))/R

We obtain that

$$ \ddot{\theta} = a \sin(\theta)$$

We can use the chain rule $\ddot{\theta} = \frac{d\dot{\theta}}{d\theta} \frac{d\theta}{dt} = \dot{\theta} \frac{d\dot{\theta}}{d\theta}$

$$ \int \dot{\theta} d\dot{\theta} = \int a \sin(\theta) d\theta $$

$$ \frac{1}{2}\dot{\theta}^2 = - a \cos(\theta) + c $$

$$ \dot{\theta}^2 = -2 a \cos(\theta) + c $$

Solve for $c$, such that the intial conditions are $\theta = \dot{\theta} = 0$

$$ c = 2a $$

$$ \dot{\theta}^2 = 2a(1 - \cos(\theta)) $$

In [12]:
a = g / R
a

g/R

We can substitute $\dot{\theta}^2$ inside the $r$ equation.

In [13]:
lm_eq = system[0].subs({the_d**2: 2 * a * (1 - smp.cos(the))}).simplify()
lm_eq

\lambda + 1.0*g*m*(3*cos(\theta(t)) - 2)

In [14]:
lm_eq = smp.solve(lm_eq, lm)[0]
smp.Eq(lm, lm_eq)

Eq(\lambda, g*m*(2.0 - 3.0*cos(\theta(t))))

This is force of constraint along the $r$ direction. We need to find when this equation equals zero

In [15]:
smp.Eq(0, lm_eq).simplify()

Eq(g*m*(2.0 - 3.0*cos(\theta(t))), 0)

In [16]:
smp.solve(lm_eq, the)

[0.841068670567930, 5.44211663661166]